In [ ]:
!pip install -q spacy
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 87.0 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [4]:
# =========================================================
# PRESCRIPTION NER PIPELINE
# =========================================================

import json
import re
import random
import spacy
from spacy.training import offsets_to_biluo_tags
from spacy.training.example import Example
from spacy.util import minibatch, compounding


# =========================================================
# DICTIONARIES & REGEX
# =========================================================

PURPOSE_WORDS = [
    "allergy", "pain", "fever", "diarrhoea", "cough",
    "cold", "infection", "vomiting", "acidity"
]

NOTE_WORDS = [
    "after food", "before food", "at bedtime",
    "morning only", "night only", "empty stomach"
]

# NOTE: \s* instead of \s+ to handle "10mg" and "10 mg" both
STRENGTH_RE = r'\b\d+(\.\d+)?(\s*(mg|ml|mcg|g|iu))(\s*/\s*\d+(\.\d+)?\s*(mg|ml|mcg|g|iu))?\b'
DOSAGE_RE   = r'\b\d+\s*(tablet|tab|capsule|cap|drop(?:s)?)\b'
FREQ_RE     = r'\b(OD|BD|TID|TDS|QID|HS|SOS)\b'
DURATION_RE = r'\b(\d+\s*(days?|weeks?|months?)|x\d+[dwm]|\d+[dwm])\b'

MEDICINE_PREFIX_RE = r'^\s*(?:T/|Tab\.?|Tablet|Caps?\.?|Capsule|Syp\.?|Syr\.?|Syrup|Inj\.?)\s*'


# =========================================================
# CORE HELPERS
# =========================================================

def is_overlapping(start, end, entities):
    for s, e, _ in entities:
        if start < e and end > s:
            return True
    return False


def snap_to_token_boundaries(doc, start, end):
    """Snap char offsets to token boundaries using spaCy."""
    span = doc.char_span(start, end, alignment_mode="expand")
    if span is None or len(span) == 0:
        return None
    return span.start_char, span.end_char


def is_valid_example(doc, entities):
    """
    Returns True only if ALL entities align perfectly to token boundaries.
    Checks BEFORE creating Example so no W030 warning is ever emitted.
    """
    tags = offsets_to_biluo_tags(doc, entities)
    return '-' not in tags


# =========================================================
# WEAK LABELLER
# =========================================================

def generate_training_example(text, tokenizer):
    doc      = tokenizer(text)
    entities = []

    def try_add(start, end, label):
        snapped = snap_to_token_boundaries(doc, start, end)
        if snapped is None:
            return
        s, e = snapped
        if not is_overlapping(s, e, entities):
            entities.append((s, e, label))

    # STRENGTH  (handles "250 mg/5 ml" as well)
    for m in re.finditer(STRENGTH_RE, text, re.I):
        try_add(m.start(), m.end(), "STRENGTH")

    # DOSAGE
    for m in re.finditer(DOSAGE_RE, text, re.I):
        try_add(m.start(), m.end(), "DOSAGE")

    # FREQUENCY
    for m in re.finditer(FREQ_RE, text, re.I):
        try_add(m.start(), m.end(), "FREQUENCY")

    # DURATION
    for m in re.finditer(DURATION_RE, text, re.I):
        try_add(m.start(), m.end(), "DURATION")

    # PURPOSE
    for word in PURPOSE_WORDS:
        for m in re.finditer(r'\b' + re.escape(word) + r'\b', text, re.I):
            try_add(m.start(), m.end(), "PURPOSE")

    # NOTES
    for note in NOTE_WORDS:
        for m in re.finditer(re.escape(note), text, re.I):
            try_add(m.start(), m.end(), "NOTES")

    # MEDICINE — strip prefix, grab first alpha-token run before any digit
    stripped      = re.sub(MEDICINE_PREFIX_RE, '', text, flags=re.I)
    prefix_offset = len(text) - len(stripped)
    leading_ws    = len(stripped) - len(stripped.lstrip())

    med_match = re.match(
        r'([A-Za-z][A-Za-z0-9\-]*(?:\s+[A-Za-z][A-Za-z0-9\-]*)*)',
        stripped.strip()
    )
    if med_match:
        raw_start = prefix_offset + leading_ws + med_match.start()
        raw_end   = prefix_offset + leading_ws + med_match.end()
        try_add(raw_start, raw_end, "MEDICINE")

    return (text, {"entities": entities})


# =========================================================
# LOAD DATASET
# =========================================================

INPUT_FILE = "/content/prescription_raw_text_only.json"

with open(INPUT_FILE, "r", encoding="utf-8") as f:
    raw_data = json.load(f)

print(f"Loaded {len(raw_data)} records")


# =========================================================
# BUILD MODEL SKELETON
# =========================================================

nlp = spacy.blank("en")
ner = nlp.add_pipe("ner", last=True)

LABELS = ["MEDICINE", "STRENGTH", "DOSAGE", "FREQUENCY", "DURATION", "PURPOSE", "NOTES"]
for label in LABELS:
    ner.add_label(label)

tokenizer = nlp.tokenizer


# =========================================================
# BUILD & VALIDATE TRAINING DATA  (zero W030 warnings)
# =========================================================

TRAIN_DATA = []
skipped    = 0

for row in raw_data:
    text = row["raw_text"].strip()
    if not text:
        continue

    example_text, annotations = generate_training_example(text, tokenizer)
    entities                  = annotations["entities"]

    # ── Validate BEFORE touching Example.from_dict ──────
    doc = nlp.make_doc(example_text)

    if not is_valid_example(doc, entities):
        # Try to salvage: keep only the perfectly-aligned subset
        good_entities = []
        for ent in entities:
            subset_tags = offsets_to_biluo_tags(doc, [ent])
            if '-' not in subset_tags:
                good_entities.append(ent)

        # Re-check the salvaged set for internal overlaps
        final_entities = []
        for ent in good_entities:
            s, e, lbl = ent
            if not is_overlapping(s, e, final_entities):
                final_entities.append(ent)

        if not final_entities:
            skipped += 1
            continue

        annotations = {"entities": final_entities}

        # Final sanity check on salvaged set
        if not is_valid_example(doc, final_entities):
            skipped += 1
            continue

    TRAIN_DATA.append((example_text, annotations))

print(f"Valid training examples : {len(TRAIN_DATA)}")
print(f"Skipped entirely        : {skipped}")


# =========================================================
# TRAIN
# =========================================================

EPOCHS     = 20
BATCH_SIZE = compounding(4.0, 32.0, 1.001)
DROP       = 0.3

with nlp.disable_pipes(*[p for p in nlp.pipe_names if p != "ner"]):

    optimizer = nlp.initialize()

    for epoch in range(EPOCHS):
        random.shuffle(TRAIN_DATA)
        losses = {}

        for batch in minibatch(TRAIN_DATA, size=BATCH_SIZE):
            examples = []
            for text, annotations in batch:
                doc = nlp.make_doc(text)
                examples.append(Example.from_dict(doc, annotations))
            nlp.update(examples, drop=DROP, losses=losses)

        print(f"Epoch {epoch+1:>2}/{EPOCHS}  NER loss: {losses.get('ner', 0):.4f}")


# =========================================================
# SAVE MODEL
# =========================================================

nlp.to_disk("prescription_ner_model")
print("\nModel saved → prescription_ner_model/")


# =========================================================
# INFERENCE
# =========================================================

def get_form_from_dosage(dosage: str) -> str:
    d = dosage.lower()
    if "tablet" in d or "tab" in d:  return "tablet"
    if "capsule" in d or "cap" in d: return "capsule"
    if "drop"    in d:               return "drops"
    if "ml"      in d:               return "liquid"
    return ""


def extract_entities(text: str) -> dict:
    doc    = nlp(text)
    result = {
        "raw_text":      text,
        "medicine_name": "",
        "form":          "",
        "strength":      "",
        "dosage":        "",
        "frequency":     "",
        "duration":      "",
        "purpose":       "",
        "notes":         ""
    }
    for ent in doc.ents:
        if   ent.label_ == "MEDICINE":  result["medicine_name"] = ent.text
        elif ent.label_ == "STRENGTH":  result["strength"]      = ent.text
        elif ent.label_ == "DOSAGE":
            result["dosage"] = ent.text
            result["form"]   = get_form_from_dosage(ent.text)
        elif ent.label_ == "FREQUENCY": result["frequency"] = ent.text
        elif ent.label_ == "DURATION":  result["duration"]  = ent.text
        elif ent.label_ == "PURPOSE":   result["purpose"]   = ent.text
        elif ent.label_ == "NOTES":     result["notes"]     = ent.text
    return result


# =========================================================
# PREDICT & SAVE
# =========================================================

results = [extract_entities(row["raw_text"]) for row in raw_data]

with open("structured_output.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=4)

print("\nSample Output:\n")
print(json.dumps(results[:5], indent=4))


# =========================================================
# FILL-RATE SUMMARY
# =========================================================

total  = len(results)
fields = ["medicine_name", "form", "strength", "dosage",
          "frequency", "duration", "purpose", "notes"]
counts = {f: sum(1 for r in results if r[f]) for f in fields}

print("\n================ Fill-Rate Summary ================\n")
print(f"Total Records : {total}\n")
for f in fields:
    print(f"{f:<20}: {counts[f]:>4} / {total}  ({counts[f]/total*100:.1f}%)")

overall = sum(counts.values()) / (total * len(fields))
print(f"\nOverall Fill Rate: {overall*100:.2f}%")

Loaded 10000 records
Valid training examples : 10000
Skipped entirely        : 0
Epoch  1/20  NER loss: 4219.8193
Epoch  2/20  NER loss: 684.4696
Epoch  3/20  NER loss: 431.1518
Epoch  4/20  NER loss: 303.8210
Epoch  5/20  NER loss: 197.8323
Epoch  6/20  NER loss: 161.5360
Epoch  7/20  NER loss: 163.8465
Epoch  8/20  NER loss: 161.3819
Epoch  9/20  NER loss: 135.5634
Epoch 10/20  NER loss: 104.8304
Epoch 11/20  NER loss: 111.0561
Epoch 12/20  NER loss: 125.9180
Epoch 13/20  NER loss: 108.5416
Epoch 14/20  NER loss: 79.8700
Epoch 15/20  NER loss: 108.1198
Epoch 16/20  NER loss: 89.9772
Epoch 17/20  NER loss: 96.6555
Epoch 18/20  NER loss: 59.5435
Epoch 19/20  NER loss: 73.9134
Epoch 20/20  NER loss: 69.3700

Model saved → prescription_ner_model/

Sample Output:

[
    {
        "raw_text": "Fexofenadin 120 mg OD allergy 7 days",
        "medicine_name": "Fexofenadin",
        "form": "",
        "strength": "120 mg",
        "dosage": "",
        "frequency": "OD",
        "duration": "